# MODI-HChar Handwritten Character Recognition
## Reimplementation: DenseNet121 | EfficientNet-B0 | EfficientNet-B1
### Tasks: Numerals (A) · Vowels (B) · Consonants (C)

## Phase 0: Environment Setup & Imports

In [2]:
! pip install torch torchvision timm scikit-learn matplotlib seaborn pandas tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached markupsafe-3.0.3-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   - -------------------------------------- 4.7/123.0 MB 22.0 MB/s eta 0:00:06
   -- ------------------------------------- 8.9/123.0 MB 21.3 MB/s eta 0:00:06
   ---- ----------------------------------- 13.4/123.0 MB 21.5 MB/s eta 0:00:06
   ----- ---------------------------------- 17.6/123.0 MB 21.3 MB/s eta 0:00:05
   ------- -------------------------------- 22.5/123.0 MB 21.6 MB/s eta 0:00:05
   -------- ------------------------------- 26.7/123.0 MB 21.5 MB/s eta 0:00:0


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Install dependencies if needed
# !pip install torch torchvision timm scikit-learn matplotlib seaborn pandas tqdm

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from torchvision import transforms, datasets, models
import timm  # for EfficientNet-B0 / B1

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

c:\Users\KJSCE\Desktop\LY_Project_C36\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Phase 1: Dataset Preparation

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# UPDATE this path to where MODI-HChar dataset is stored on your system.
# Expected structure:
#   DATASET_ROOT/
#       class_00/  (or class name folder)
#           img1.png
#           ...
#       class_01/
#       ...
# If train/test are pre-split:
#   DATASET_ROOT/train/<class>/<images>
#   DATASET_ROOT/test/<class>/<images>
# ─────────────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path('./MODI_HChar/split')   # <── CHANGE THIS

# Class index ranges (0-based, adjust if your folder naming differs)
# 57 classes total: 0-9 numerals, 10-21 vowels, 22-56 consonants
TASK_CONFIG = {
    'A_Numerals':    {'num_classes': 10,  'class_range': range(0, 10)},
    'B_Vowels':      {'num_classes': 12,  'class_range': range(10, 22)},
    'C_Consonants':  {'num_classes': 35,  'class_range': range(22, 57)},
}

# Training hyper-parameters per model (Table IV)
MODEL_CONFIG = {
    'DenseNet121':    {'epochs': 50, 'lr': 1e-4, 'batch_size': 32},
    'EfficientNet-B0':{'epochs': 35, 'lr': 1e-4, 'batch_size': 32},
    'EfficientNet-B1':{'epochs': 45, 'lr': 1e-4, 'batch_size': 32},
}

IMG_SIZE   = 224   # standard input size for all three architectures
VAL_SPLIT  = 0.15  # 15 % of training data used for validation
TEST_SPLIT = 0.15  # 15 % used for test (if no pre-split available)

print('Configuration loaded.')
print(f'Tasks   : {list(TASK_CONFIG.keys())}')
print(f'Models  : {list(MODEL_CONFIG.keys())}')

Configuration loaded.
Tasks   : ['A_Numerals', 'B_Vowels', 'C_Consonants']
Models  : ['DenseNet121', 'EfficientNet-B0', 'EfficientNet-B1']


In [11]:
# ── Dataset sanity check ─────────────────────────────────────────────────────
def inspect_dataset(root: Path):
    """Print class counts for the root directory."""
    if not root.exists():
        print(f'[WARNING] Path does not exist: {root}')
        return
    classes = sorted([d for d in root.iterdir() if d.is_dir()])
    total = 0
    print(f'Found {len(classes)} class folders in {root}')
    for c in classes[:5]:          # preview first 5
        n = len(list(c.glob('*.*')))
        total += n
        print(f'  {c.name}: {n} images')
    if len(classes) > 5:
        print(f'  ... ({len(classes)-5} more classes)')
    return classes

all_classes = inspect_dataset(DATASET_ROOT)

Found 2 class folders in MODI_HChar\split
  test: 0 images
  train: 0 images


## Phase 2: Data Preprocessing & Augmentation

In [ ]:
# ── Transforms ───────────────────────────────────────────────────────────────
class AddGaussianNoise(object):
    """Simulate historical document degradation via Gaussian noise."""
    def __init__(self, mean=0.0, std=0.05):
        self.mean = mean
        self.std  = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

    def __repr__(self):
        return f'AddGaussianNoise(mean={self.mean}, std={self.std})'


# Training transforms: normalization + augmentation (rotation, scaling, noise)
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # convert 1-ch → 3-ch
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=15),          # Rotation augmentation
    transforms.RandomAffine(
        degrees=0,
        scale=(0.85, 1.15),                         # Scaling augmentation
        translate=(0.05, 0.05)
    ),
    transforms.RandomHorizontalFlip(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    AddGaussianNoise(mean=0.0, std=0.03),           # Noise augmentation
])

# Validation / test transforms: normalization only (no augmentation)
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Transforms defined:')
print('  Train:', train_transform)
print('  Val/Test:', val_transform)

In [ ]:
# ── Task-specific subset dataset ─────────────────────────────────────────────
class TaskSubset(Dataset):
    """
    Wraps an ImageFolder and exposes only the classes in `class_indices`.
    Re-maps labels to 0-based within the task.
    """
    def __init__(self, full_dataset, class_indices, transform=None):
        self.transform      = transform
        self.class_indices  = sorted(class_indices)
        self.label_map      = {orig: new
                               for new, orig in enumerate(self.class_indices)}
        # Filter samples belonging to the target classes
        self.samples = [
            (path, self.label_map[label])
            for path, label in full_dataset.samples
            if label in self.label_map
        ]
        self.classes = [full_dataset.classes[i] for i in self.class_indices]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        from PIL import Image
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


def build_loaders(root: Path, class_range, batch_size: int, seed: int = 42):
    """
    Build train, validation and test DataLoaders for a given task.
    Handles both pre-split (root/train, root/test) and flat directory layouts.
    """
    # ── detect layout ────────────────────────────────────────────────────────
    has_split = (root / 'train').exists() and (root / 'test').exists()

    if has_split:
        full_train = datasets.ImageFolder(root / 'train')
        full_test  = datasets.ImageFolder(root / 'test')
        train_ds = TaskSubset(full_train, class_range, train_transform)
        test_ds  = TaskSubset(full_test,  class_range, val_transform)

        # carve out validation from training
        n_val   = int(len(train_ds) * VAL_SPLIT)
        n_train = len(train_ds) - n_val
        generator = torch.Generator().manual_seed(seed)
        train_ds, val_ds = random_split(train_ds, [n_train, n_val],
                                        generator=generator)
        # val_ds must use val_transform — wrap it
        val_ds.dataset.transform = val_transform

    else:
        full_ds = datasets.ImageFolder(root)
        task_ds = TaskSubset(full_ds, class_range, None)   # transform set later

        n_total = len(task_ds)
        n_test  = int(n_total * TEST_SPLIT)
        n_val   = int(n_total * VAL_SPLIT)
        n_train = n_total - n_val - n_test
        generator = torch.Generator().manual_seed(seed)
        train_idx, val_idx, test_idx = random_split(
            range(n_total), [n_train, n_val, n_test], generator=generator)

        class SplitWrapper(Dataset):
            def __init__(self, base, indices, tfm):
                self.base, self.indices, self.tfm = base, indices, tfm
            def __len__(self): return len(self.indices)
            def __getitem__(self, i):
                img, lbl = self.base[self.indices[i]]
                # img is a PIL image here (transform=None on base)
                if self.tfm: img = self.tfm(img)
                return img, lbl

        # Override __getitem__ in TaskSubset to return PIL when transform=None
        def _getitem_pil(self_inner, idx):
            path, label = self_inner.samples[idx]
            from PIL import Image
            image = Image.open(path).convert('RGB')
            if self_inner.transform:
                image = self_inner.transform(image)
            return image, label
        TaskSubset.__getitem__ = _getitem_pil

        train_ds = SplitWrapper(task_ds, train_idx.indices, train_transform)
        val_ds   = SplitWrapper(task_ds, val_idx.indices,   val_transform)
        test_ds  = SplitWrapper(task_ds, test_idx.indices,  val_transform)

    # ── DataLoaders ──────────────────────────────────────────────────────────
    num_workers = min(4, os.cpu_count() or 1)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True)

    print(f'  Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
    return train_loader, val_loader, test_loader


print('DataLoader builder ready.')

## Phase 3: Model Architecture Setup

In [ ]:
def build_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Returns a pre-trained model with its classifier head replaced
    to match `num_classes`.
    """
    if model_name == 'DenseNet121':
        # 121-layer dense connectivity: each layer receives feature maps
        # from ALL preceding layers → strong feature reuse, no vanishing gradient
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    elif model_name == 'EfficientNet-B0':
        # 237-layer compound-scaled network (baseline)
        # Uniformly scales depth/width/resolution with fixed coefficient φ
        model = timm.create_model('efficientnet_b0', pretrained=True,
                                   num_classes=num_classes)

    elif model_name == 'EfficientNet-B1':
        # 329-layer scaled-up B0: higher depth & resolution for complex scripts
        model = timm.create_model('efficientnet_b1', pretrained=True,
                                   num_classes=num_classes)
    else:
        raise ValueError(f'Unknown model: {model_name}')

    return model.to(DEVICE)


# ── Quick architecture summary ───────────────────────────────────────────────
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for mname in MODEL_CONFIG:
    m = build_model(mname, num_classes=10)
    print(f'{mname:20s} → trainable params: {count_parameters(m):,}')
    del m

## Phase 4: Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return (running_loss / total,
            100.0 * correct / total,
            np.array(all_preds),
            np.array(all_labels))


def train_model(model_name, task_name, num_classes,
                train_loader, val_loader, test_loader):
    cfg       = MODEL_CONFIG[model_name]
    epochs    = cfg['epochs']
    lr        = cfg['lr']

    model     = build_model(model_name, num_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    history   = defaultdict(list)
    best_val_acc = 0.0
    best_path = f'/tmp/best_{model_name}_{task_name}.pth'

    print(f'\n{'='*60}')
    print(f'  Model: {model_name}  |  Task: {task_name}  |  Epochs: {epochs}')
    print(f'{'='*60}')

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc           = train_one_epoch(model, train_loader,
                                                     criterion, optimizer, scaler)
        vl_loss, vl_acc, _, _     = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch [{epoch:3d}/{epochs}]  '
                  f'Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc:.2f}%  '
                  f'Val Loss: {vl_loss:.4f}  Val Acc: {vl_acc:.2f}%')

    # Load best checkpoint for testing
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    _, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
    print(f'\n  Best Val Acc : {best_val_acc:.2f}%')
    print(f'  Test Acc     : {test_acc:.2f}%')

    return model, history, test_preds, test_labels

print('Training utilities defined.')

## Phase 5: Evaluation Metrics (Table III)

In [ ]:
def compute_metrics(y_true, y_pred, num_classes):
    """
    Compute all five metrics from Table III:
        Accuracy, Specificity, Sensitivity (Recall), Precision, F1
    Uses macro-averaging for multi-class problems.
    """
    avg = 'macro'

    accuracy    = accuracy_score(y_true, y_pred) * 100
    precision   = precision_score(y_true, y_pred, average=avg, zero_division=0) * 100
    sensitivity = recall_score(y_true, y_pred, average=avg, zero_division=0) * 100   # Recall
    f1          = f1_score(y_true, y_pred, average=avg, zero_division=0) * 100

    # Specificity = TN / (TN + FP)  — computed per class then averaged
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    spec_per_class = []
    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FP - FN
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        spec_per_class.append(spec)
    specificity = np.mean(spec_per_class) * 100

    return {
        'Accuracy (%)':    round(accuracy, 4),
        'Specificity (%)': round(specificity, 4),
        'Sensitivity (%)': round(sensitivity, 4),   # Recall
        'Precision (%)':   round(precision, 4),
        'F1 Score (%)':    round(f1, 4),
    }


def plot_training_curves(history, model_name, task_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='coral')
    axes[0].set_title(f'{model_name} | {task_name} — Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='Train Acc', color='steelblue')
    axes[1].plot(epochs, history['val_acc'],   label='Val Acc',   color='coral')
    axes[1].set_title(f'{model_name} | {task_name} — Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'/tmp/curves_{model_name}_{task_name}.png', dpi=120)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, model_name, task_name):
    cm = confusion_matrix(y_true, y_pred)
    fig_size = max(8, len(class_names) // 2)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'Confusion Matrix — {model_name} | {task_name}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'/tmp/cm_{model_name}_{task_name}.png', dpi=120)
    plt.show()


print('Evaluation utilities defined.')

## Phase 6: Full Experiment Runner

In [ ]:
# ── Main experiment loop ─────────────────────────────────────────────────────
# Stores all results for final comparison table
all_results = []   # list of dicts

for task_name, task_cfg in TASK_CONFIG.items():
    num_classes  = task_cfg['num_classes']
    class_range  = list(task_cfg['class_range'])

    print(f'\n{"#"*70}')
    print(f'  TASK: {task_name}  ({num_classes} classes)')
    print(f'{"#"*70}')

    # Build data loaders (batch size taken from first model; same across models)
    first_bs = list(MODEL_CONFIG.values())[0]['batch_size']
    train_loader, val_loader, test_loader = build_loaders(
        DATASET_ROOT, class_range, first_bs)

    for model_name in MODEL_CONFIG:
        model, history, test_preds, test_labels = train_model(
            model_name, task_name, num_classes,
            train_loader, val_loader, test_loader
        )

        # Compute metrics
        metrics = compute_metrics(test_labels, test_preds, num_classes)

        # Store for comparison table
        row = {'Model': model_name, 'Task': task_name}
        row.update(metrics)
        all_results.append(row)

        # Plot training curves
        plot_training_curves(history, model_name, task_name)

        # Confusion matrix — only for numerals/vowels (manageable size)
        if num_classes <= 12:
            plot_confusion_matrix(
                test_labels, test_preds,
                [str(i) for i in range(num_classes)],
                model_name, task_name
            )

        print(f'\n  Metrics for {model_name} on {task_name}:')
        for k, v in metrics.items():
            print(f'    {k:<20s}: {v:.4f} %')

print('\n✓ All experiments complete.')

## Phase 6: Comparative Analysis (Table III Reproduction)

In [ ]:
# ── Results DataFrame ────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df = results_df.set_index(['Task', 'Model'])

pd.set_option('display.float_format', '{:.4f}'.format)
print('\n=== FULL RESULTS TABLE (Table III Reproduction) ===')
print(results_df.to_string())

In [ ]:
# ── Bar chart: Accuracy comparison across tasks and models ───────────────────
pivot_acc = results_df['Accuracy (%)'].unstack('Model')

ax = pivot_acc.plot(
    kind='bar', figsize=(12, 6),
    color=['#4C72B0', '#DD8452', '#55A868'],
    edgecolor='black', linewidth=0.6
)
ax.set_title('Test Accuracy Comparison — All Models & Tasks', fontsize=14, fontweight='bold')
ax.set_xlabel('Task', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_ylim(0, 105)
ax.legend(title='Model', fontsize=10)
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3, fontsize=8)
plt.xticks(rotation=0, fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Radar / Spider chart: all 5 metrics per task for best model ──────────────
from matplotlib.patches import FancyArrowPatch

metrics_cols = ['Accuracy (%)', 'Specificity (%)', 'Sensitivity (%)',
                'Precision (%)', 'F1 Score (%)']

fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw=dict(polar=True))
labels = [m.replace(' (%)', '') for m in metrics_cols]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

colors = {'DenseNet121': '#4C72B0', 'EfficientNet-B0': '#DD8452',
          'EfficientNet-B1': '#55A868'}

for ax, (task_name, _) in zip(axes, TASK_CONFIG.items()):
    for model_name in MODEL_CONFIG:
        try:
            row = results_df.loc[(task_name, model_name), metrics_cols]
        except KeyError:
            continue
        values = row.values.tolist()
        values += values[:1]
        ax.plot(angles, values, color=colors[model_name], linewidth=2,
                label=model_name)
        ax.fill(angles, values, color=colors[model_name], alpha=0.15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=7)
    ax.set_title(task_name, fontsize=12, fontweight='bold', pad=15)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.suptitle('5-Metric Radar Chart per Task', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/radar_metrics.png', dpi=150)
plt.show()

In [ ]:
# ── Export results to CSV ────────────────────────────────────────────────────
out_path = '/tmp/MODI_HChar_results.csv'
results_df.to_csv(out_path)
print(f'Results saved to: {out_path}')
print('\n=== SUMMARY ===')
print(results_df.groupby('Model')['Accuracy (%)'].mean().sort_values(ascending=False))